# 17 · ประเมิน KNN และ Linear Regression กับข้อมูลกลุ่มและเวลา

[กลับสารบัญ](../../README.md) · [Notebook](lesson.ipynb) · [Python](lesson.py)

บทนี้เลือกการแบ่งข้อมูลให้ตรงกับงาน **ทำนายคนใหม่** หรือ **ทำนายอนาคต**

**ใช้สิ่งที่เรียนแล้ว:** train/test, CV, KNN และ regression/MAE

**Group split** และ **Time-based split** เป็นวิธีแบ่งข้อมูลเพื่อประเมินโมเดล ไม่ใช่อัลกอริทึมทำนาย จุดประสงค์คือให้ข้อมูลที่ใช้ตรวจสอบต่างจากชุดฝึกในลักษณะเดียวกับงานจริง

| งานที่ต้องการประเมิน | วิธีแบ่ง | จุดเด่นและข้อจำกัด |
|---|---|---|
| ทำนายคนที่ไม่เคยอยู่ในชุดฝึก | เก็บทุกแถวของคนเดียวกันไว้ฝั่งเดียว | ทดสอบกับคนใหม่จริง ๆ แต่ถ้ามีคนทดสอบน้อย คะแนนอาจเปลี่ยนมากตามคนที่ถูกเลือก |
| ทำนายช่วงเวลาถัดไป | ฝึกอดีต ตรวจช่วงหลัง | สอดคล้องลำดับเวลาจริง แต่รูปแบบในอนาคตอาจเปลี่ยนจากช่วงที่ตรวจ |

เลือกวิธีแบ่งตามการใช้งานจริง ไม่เลือกจากวิธีที่ให้คะแนนสูงที่สุด

## โมเดลที่ใช้ทดลองเรื่องการแบ่งข้อมูล

**กรณีคนเดิมหลายแถว:** ใช้ **KNN** จาก [บท 03](../03-first-classifier/README.md) ผ่าน `KNeighborsClassifier(n_neighbors=3)` เลือกสามตัวอย่างที่ใกล้ที่สุดแล้วโหวตคลาส 0/1 ถ้าคนเดิมหลุดไปทั้ง train/test ตัวอย่างใกล้ที่สุดอาจเป็นข้อมูลของคนเดิม คะแนนจึงอาจสูงโดยไม่ได้แสดงความสามารถกับคนใหม่

**กรณีข้อมูลเวลา:** ใช้ **Linear Regression** จาก [บท 05](../05-linear-regression/README.md) รับค่าก่อนหน้าหนึ่งค่าเป็น X และเรียนสมการ `ค่าถัดไป ≈ b + w × ค่าก่อนหน้า` จากช่วงอดีต แล้วเทียบกับ **persistence baseline** ที่ทายค่าถัดไปให้เท่าค่าล่าสุด เช่น ล่าสุดเป็น 12 ก็ทาย 12 โดยไม่เรียน w/b

KNN ไวต่อระยะทางและตัวอย่างอ้างอิง ส่วน Linear Regression ใช้ความสัมพันธ์เชิงเส้น การแบ่งกลุ่มหรือเวลาไม่ได้เปลี่ยนข้อจำกัดของโมเดล แต่ช่วยให้การประเมินตรงกับงานที่ต้องการใช้จริง

![กรณีคนใหม่ใช้ KNN สามเพื่อนบ้านทายคลาสหลังแยกคนออกทั้งกลุ่ม ส่วนกรณีเวลาใช้ Linear Regression จากค่าก่อนหน้าและเทียบ persistence baseline](../../assets/figures/17-time-and-groups-models.png)

**อ่านภาพ:** แถวบนใช้ KNN ตอบคลาส ส่วนแถวล่างใช้ Linear Regression ตอบตัวเลข โดยค่า 12 เป็นตัวอย่างสมมติของค่าล่าสุด ทั้งสองกรณีต้องฝึกเฉพาะข้อมูลที่อนุญาตก่อนประเมิน · [เปิดภาพ SVG](../../assets/figures/17-time-and-groups-models.svg)

## คนเดิมหลายแถว: แบ่งตามกลุ่ม

ข้อมูลจำลองมี 60 คน คนละ 5 แถว ข้อมูลนำเข้า (features) ทั้งห้าแถวของคนเดียวกันมีค่าใกล้กัน และใช้คลาส (class label) เดียวกันที่สุ่มกำหนดให้คนนั้น

**ปัญหาของการสุ่มแถว:** คนเดิมอาจอยู่ทั้ง train และ test โมเดลจึงอาจจำคนเดิมได้ แทนที่จะวัดความสามารถกับคนใหม่

**Group split:** เก็บทุกแถวของคนหนึ่งไว้ใน train หรือ test ฝั่งใดฝั่งหนึ่ง เช่น คน ก มี 5 แถว หากเลือกคน ก เป็น test ต้องนำทั้ง 5 แถวไปด้วย รหัสคนใช้จัดแบ่งข้อมูล ไม่ส่งให้โมเดลทำนาย

คลาสถูกสุ่มกำหนดให้แต่ละคน จึงไม่มีความสัมพันธ์กับข้อมูลนำเข้าที่จะใช้ทายคลาสของคนใหม่ได้อย่างสม่ำเสมอ และเมื่อทดสอบกับคนจำนวนน้อย คะแนนยิ่งแกว่งได้มาก

## ข้อมูลเวลา: ฝึกอดีต ตรวจช่วงถัดไป

**TimeSeriesSplit:** แต่ละรอบใช้ข้อมูลช่วงแรกเป็น train และช่วงถัดมาเป็น validation รอบต่อไปเพิ่มข้อมูลช่วงที่ผ่านไปแล้วเข้า train แล้วเลื่อนไปตรวจช่วงที่ใหม่กว่า จึงไม่ใช้อนาคตฝึกเพื่อย้อนมาทายอดีต

**Gap:** ช่วงที่เว้นระหว่างข้อมูลฝึกกับช่วงตรวจสอบ ขนาดที่เหมาะขึ้นอยู่กับช่วงเวลาที่ข้อมูลนำเข้าครอบคลุม และเวลาที่ทราบค่าจริงของสิ่งที่ต้องการทำนาย

**Lag:** ค่าจากเวลาก่อนหน้า เช่น series `[10, 12, 11, 15]` ใช้ค่าก่อนหน้าเป็นข้อมูลนำเข้า แล้วใช้ค่าถัดไปเป็นค่าที่ต้องการทำนาย จึงได้คู่ `10→12, 12→11, 11→15`

**Persistence baseline:** ทายค่าถัดไปให้เท่ากับค่าล่าสุด เพื่อใช้เทียบกับโมเดล

**One-step-ahead:** ทำนายจุดถัดไปเพียงหนึ่งจุดจากค่าล่าสุดที่รู้จริง เช่น ใช้ค่าจริงที่เวลา 1 ทายเวลา 2 เมื่อได้ค่าจริงที่เวลา 2 แล้วจึงใช้ค่านั้นทายเวลา 3 นี่คือรูปแบบที่บทนี้ประเมิน

หากทำนายหลายก้าวจากวันเริ่มเดียว จะยังไม่รู้ค่าจริงระหว่างทาง จึงนำค่าจริงเหล่านั้นมาใช้แบบโค้ดนี้ไม่ได้

คะแนนในส่วนข้อมูลเวลามาจาก validation ของแต่ละรอบ ยังไม่ได้กันช่วงท้ายอีกชุดไว้ทดสอบครั้งสุดท้าย หากนำคะแนนเหล่านี้ไปเลือกโมเดล ควรเก็บช่วงท้ายที่ยังไม่ใช้ตัดสินใจไว้ตรวจผลด้วย ชุดที่กันไว้นี้เรียกว่า **final future holdout**

## ลงมือทำ

สร้าง Notebook ใหม่แล้วรัน **Cell 1–2 ตามลำดับ** หรือเปิด [Notebook ของบทนี้](lesson.ipynb) ใช้ kernel เดิมได้เลย อ่านคำอธิบายใต้โค้ดแต่ละช่วงประกอบผลที่ได้

<!-- CODE:CELLS -->



### Cell 1 — เปรียบเทียบการสุ่มแถวกับการแยกคน


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit, TimeSeriesSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(42)
groups = np.repeat(np.arange(60), 5)
user_signal = rng.normal(size=(60, 3))
X = user_signal[groups] + rng.normal(0, 0.05, (len(groups), 3))
user_label = rng.integers(0, 2, size=60)
y = user_label[groups]

# การสุ่มแถวใช้เทียบให้เห็นการปะปนของคนเดิม ไม่ใช้เลือกวิธีประเมินจากคะแนนสูง
row_train, row_test = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42)
row_model = KNeighborsClassifier(n_neighbors=3).fit(X[row_train], y[row_train])
shared_users = set(groups[row_train]) & set(groups[row_test])
print("Random row split - shared users:", len(shared_users))
print(f"Random row-split accuracy: {row_model.score(X[row_test], y[row_test]):.3f}")

split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(split.split(X, y, groups))
assert set(groups[train_idx]).isdisjoint(groups[test_idx])
model = KNeighborsClassifier(n_neighbors=3).fit(X[train_idx], y[train_idx])
print("Group split - shared users:", len(set(groups[train_idx]) & set(groups[test_idx])))
print("Held-out user groups:", len(np.unique(groups[test_idx])))
print(f"New-user accuracy (random user labels): {model.score(X[test_idx], y[test_idx]):.3f}")



**อ่านโค้ด**

- `np.repeat(np.arange(60), 5)` สร้างรหัสคน 0–59 โดยแต่ละคนมีห้าแถว รวมเป็น 300 แถว
- `user_signal[groups]` ใช้รหัสคนเลือกค่าข้อมูลนำเข้าตั้งต้นของคนนั้น แล้วเติม noise เล็กน้อย ส่วน `user_label[groups]` ทำให้ทั้งห้าแถวของคนเดียวกันมีคลาสที่กำกับไว้เหมือนกัน
- `set(groups[row_train]) & set(groups[row_test])` หารหัสคนที่อยู่ทั้ง train และ test หลังสุ่มแถว เพื่อแสดงปัญหาคนเดิมข้ามชุด
- `next(split.split(X, y, groups))` รับคู่ index จาก `GroupShuffleSplit` รอบแรก โดยตัวแบ่งใช้ groups กันคนออกทั้งกลุ่ม
- `isdisjoint` คืน True เมื่อไม่มีรหัสคนร่วมกันระหว่าง train กับ test และ `assert` จะหยุดพร้อมข้อผิดพลาดหากเงื่อนไขนี้ไม่จริง ส่วน `np.unique(groups[test_idx])` หารหัสคนที่ไม่ซ้ำกันใน test เพื่อนำไปนับจำนวนคน
- สำหรับ KNN classifier ในบทนี้ `model.score(...)` คืน accuracy ของข้อมูลที่ส่งให้ประเมิน จึงต้องอ่านพร้อมจำนวนคนที่ซ้ำ ไม่เลือกวิธีแบ่งจากคะแนนสูงสุด

**การตั้งค่าและจำนวนแถว:** `GroupShuffleSplit(test_size=0.2)` กัน 20% ของ 60 คน จึงเป็น 12 คน × 5 แถว = test 60 แถว ไม่ได้สุ่มกันเพียงบางแถวของคนเดิม ค่า `Group split - shared users` ต้องเป็น 0



### Cell 2 — แบ่งข้อมูลตามเวลาและเทียบค่าล่าสุด


In [ ]:
t = np.arange(180)
series = 0.1 * t + np.sin(t / 6) + rng.normal(0, 0.2, len(t))
# ทำนายค่าถัดไปจากค่าก่อนหน้า ไม่ใช้ centered rolling ที่เห็นอนาคต
lag = series[:-1].reshape(-1, 1)
target = series[1:]
for fold, (train_idx, valid_idx) in enumerate(TimeSeriesSplit(n_splits=4, gap=1).split(lag), start=1):
    assert train_idx.max() < valid_idx.min()
    model = LinearRegression().fit(lag[train_idx], target[train_idx])
    pred = model.predict(lag[valid_idx])
    persistence = lag[valid_idx, 0]
    print(f"Fold {fold}: train ends={train_idx.max()}, validation starts={valid_idx.min()}, "
          f"model MAE={mean_absolute_error(target[valid_idx], pred):.3f}, "
          f"last-value MAE={mean_absolute_error(target[valid_idx], persistence):.3f}")



**อ่านโค้ด**

- `series[:-1].reshape(-1, 1)` เก็บค่าตั้งแต่ตัวแรกถึงก่อนตัวสุดท้ายไว้ใน `lag` เป็นข้อมูลนำเข้าหนึ่งคอลัมน์ (feature) ส่วน `series[1:]` เก็บค่าตั้งแต่ตัวที่สองถึงตัวสุดท้ายไว้ใน `target` เป็นค่าถัดไปที่ต้องการทำนาย
- แต่ละแถวจึงจับคู่ค่าก่อนหน้ากับค่าถัดไป โดย `lag` และ `target` มีจำนวนแถวเท่ากัน
- `TimeSeriesSplit(n_splits=4, gap=1).split(lag)` คืน index สี่รอบ ฝึกจากอดีตและเว้นหนึ่งแถวก่อนช่วง validation
- `train_idx.max() < valid_idx.min()` ตรวจว่าแถวฝึกอยู่ก่อนแถว validation
- `persistence = lag[valid_idx, 0]` ใช้ค่าล่าสุดเป็นคำทำนาย baseline แล้ววัด MAE กับ target ชุดเดียวกับโมเดล

การใช้ค่าจริงก่อนหน้าของทุกจุดเหมาะกับ one-step-ahead ตามโจทย์นี้ ไม่ใช่การทายหลายก้าวโดยยังไม่เห็นค่าจริงระหว่างทาง

**การตั้งค่าใน cell นี้:** `gap=1` เว้นหนึ่งแถวระหว่าง train กับ validation ส่วน `Fold` ที่พิมพ์ระบุรอบประเมิน และ MAE ของโมเดลกับ last-value baseline เทียบกับค่าจริงของช่วง validation เดียวกัน




## อ่านผล

- **Random row split:** อาจมีคนเดิมซ้ำทั้งสองฝั่ง
- **Group split:** ต้องมี `Shared users=0`
- **Time split:** ทุก fold ต้องมี train อยู่ก่อน validation

อ่าน MAE ของโมเดลเทียบกับ persistence baseline ในแต่ละ time fold โดยค่าต่ำกว่าดีกว่า

## แบบฝึกท้ายบท

**ลองคิด:** ทายเดือนหน้าสำหรับร้านเดิม กับร้านที่ไม่เคยเห็น ใช้การประเมินเดียวกันได้หรือไม่?

<details>
<summary>ดูเฉลย</summary>

**เฉลย:** ต้องแยกการประเมินให้ตรงกับสองโจทย์:

- **ร้านเดิม:** แบ่งตามเวลา เพื่อวัดการทำนายเดือนถัดไป
- **ร้านใหม่:** กันทั้งร้านออกจาก train พร้อมควบคุมลำดับเวลาให้ตรงการใช้งาน

รายงานผลของสองโจทย์แยกกัน

</details>

## อ้างอิง

- [Cross-validation: groups and time series](https://scikit-learn.org/1.7/modules/cross_validation.html)
- [Common pitfalls](https://scikit-learn.org/1.7/common_pitfalls.html)
- [Google MLCC: Production ML systems](https://developers.google.com/machine-learning/crash-course/production-ml-systems)
